# PTQ 官方库版：torch.ao.quantization 校准语义 + 真实 int8 存储/算子

配套文章：

- 《大模型量化算法（01）：量化器数学地基与 RTN 基线》 https://lrypcy.github.io/2026/08/23/llm-quant-00-quantizer-fundamentals-rtn/
- 《E1：RTN 与 LLM.int8()》 https://lrypcy.github.io/2026/08/24/ptq-01-rtn-llmint8/
- 《17：伪量化算子插入》 https://lrypcy.github.io/2026/08/26/llm-quant-11-fake-quant-insertion/
- 《E3：部署侧视角》 https://lrypcy.github.io/2026/09/19/llm-quant-E3-deployment-support/

**本目录与手撸目录的区别**：`rtn_llmint8/` 等目录是纯 numpy 复刻算法；本目录用的是
**PyTorch 官方量化栈**（`torch.ao.quantization` 的 observer / FakeQuantize / `quantize_per_*` /
`convert`），在**真实 resnet18 权重**上跑，回答那些手撸版回答不了的问题：

1. 官方 observer（MinMax / Histogram / MovingAverage / PerChannel）**到底选出了什么 scale**，
   与手撸的 min-max、MSE 网格最优差多少？
2. 官方 API 量出来的**磁盘体积**和**真实 int8 算子延迟**是多少？
3. torch 2.10 在 arm64 macOS 上，**官方量化栈哪些能用、哪些不能用**（这是博客 E3 需要的真实情报）。

## 运行

```bash
cd experiments/quantization/official_torchao_ptq
/Users/congyuan/Software/miniconda3/envs/torch/bin/jupyter nbconvert \
    --to notebook --execute --inplace official_torchao_ptq.ipynb
```

需要 `/Users/congyuan/Software/miniconda3/envs/torch` 环境（torch 2.10 + torchvision 0.25）。
`MODE = "smoke" | "full"` 控制规模。

In [1]:
import os, io, json, time, copy, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.ao.quantization as tq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- 关键工程细节：torch 2.10 在 arm64 macOS 上必须显式指定量化引擎 ----
torch.backends.quantized.engine = "qnnpack"
print("torch", torch.__version__, "| supported engines:", torch.backends.quantized.supported_engines,
      "| current:", torch.backends.quantized.engine)

MODE = "smoke"          # "smoke" | "full"
CFG = {
    "smoke": dict(n_calib=16, n_eval=8, batch=4, img=224, bits_grid=(8, 6, 4), reps=2),
    "full":  dict(n_calib=64, n_eval=32, batch=8, img=224, bits_grid=(8, 6, 4, 3), reps=5),
}[MODE]

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

HERE = os.getcwd()
RES = os.path.join(HERE, "results")
os.makedirs(RES, exist_ok=True)

_LINES = []
def log(msg=""):
    print(msg); _LINES.append(str(msg))

def savefig(fig, name):
    p = os.path.join(RES, name)
    fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}")
    return p

log(f"MODE={MODE} CFG={CFG}")

torch 2.10.0 | supported engines: ['qnnpack'] | current: qnnpack


MODE=smoke CFG={'n_calib': 16, 'n_eval': 8, 'batch': 4, 'img': 224, 'bits_grid': (8, 6, 4), 'reps': 2}


## 1. 模型与数据

用**真实 resnet18 权重**（ImageNet 预训练，46 MB）。没有 ImageNet 数据，所以校准/评估用合成图像
（低频结构 + 噪声，保证激活分布非退化）。**因此：体积、延迟、scale、逐层误差是真实度量；
logits 的绝对值无语义，只能做 FP32 vs PTQ 的相对比较。**

In [2]:
from torchvision.models import resnet18

CKPT = "/tmp/resnet18.pth"
model = resnet18(weights=None)
if os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location="cpu", weights_only=True))
    WEIGHTS = "ImageNet 预训练权重（真实）"
else:
    WEIGHTS = "随机初始化（未找到 /tmp/resnet18.pth）"
model.eval()
n_params = sum(p.numel() for p in model.parameters())
log(f"resnet18：{n_params/1e6:.2f} M 参数；权重来源：{WEIGHTS}")


def synth_images(n, size=224, seed=0):
    """合成图像：低频结构 + 高频噪声，避免激活分布退化成纯噪声。"""
    g = torch.Generator().manual_seed(seed)
    base = torch.randn(n, 3, 8, 8, generator=g)
    img = torch.nn.functional.interpolate(base, size=(size, size), mode="bilinear")
    img = img + 0.25 * torch.randn(n, 3, size, size, generator=g)
    return img


calib = synth_images(CFG["n_calib"], CFG["img"], seed=SEED)
evalx = synth_images(CFG["n_eval"], CFG["img"], seed=SEED + 1)


@torch.no_grad()
def logits_of(m, x, bs=None):
    bs = bs or CFG["batch"]
    outs = []
    for i in range(0, x.shape[0], bs):
        outs.append(m(x[i:i + bs]))
    return torch.cat(outs, 0)


@torch.no_grad()
def run_logits(m, x, bs=None):
    return logits_of(m, x, bs)

y_fp = run_logits(model, evalx)
log(f"FP32 logits: shape={tuple(y_fp.shape)}  absmax={float(y_fp.abs().max()):.3f}")


def state_dict_bytes(m):
    b = io.BytesIO(); torch.save(m.state_dict(), b); return len(b.getvalue())


def bench(m, x, bs=None, reps=None):
    reps = reps or CFG["reps"]; bs = bs or CFG["batch"]
    logits_of(m, x, bs)                       # warmup
    ts = []
    for _ in range(reps):
        t = time.time(); logits_of(m, x, bs); ts.append(time.time() - t)
    return min(ts) * 1000.0


fp_bytes = state_dict_bytes(model)
fp_ms = bench(model, evalx)
log(f"FP32 state_dict = {fp_bytes/1024:.1f} KB；batch 前向 = {fp_ms:.1f} ms")

resnet18：11.69 M 参数；权重来源：ImageNet 预训练权重（真实）


FP32 logits: shape=(8, 1000)  absmax=7.988


FP32 state_dict = 45737.3 KB；batch 前向 = 133.1 ms


## 实验 A：官方 observer 到底选出了什么 scale？

把同一批激活喂给四种官方 observer，把 scale 拿出来和手撸的两种基线比：

- **手撸 min-max**：`s = max|x| / 127`
- **手撸 MSE 网格最优**：在候选 scale 上网格搜索最小化 $\|x - \hat x\|^2$
- **官方 MinMaxObserver**（`per_tensor_affine`）
- **官方 HistogramObserver**（torch 默认激活 observer；它做的是 histogram 上的最优裁剪搜索，
  语义上接近"MSE 最优"而不是 min-max）
- **官方 MovingAverageMinMaxObserver**（QAT 默认）
- **官方 PerChannelMinMaxObserver**（权重 per-channel 默认）

这篇文章（01 篇 §4）的核心结论是"min-max 不是最优、MSE 最优能白捡几个 dB"——这里验证
**官方库的默认选择落在哪一侧**。

In [3]:
def manual_scale_stats(x, bits=8, symmetric=True):
    """手撸基线：min-max scale 与 MSE 网格最优 scale。

    symmetric=True  -> qint8 对称（qmin=-128, qmax=127, zp=0）
    symmetric=False -> quint8 仿射（qmin=0, qmax=255, zp 由 min 决定），与官方 affine observer 同口径
    """
    xf = x.detach().flatten().float().numpy()
    if symmetric:
        qmax, qmin, zp0 = 2 ** (bits - 1) - 1, -(2 ** (bits - 1)), 0
        s_mm = float(np.max(np.abs(xf))) / qmax
    else:
        qmin, qmax = 0, 2 ** bits - 1
        s_mm = float(xf.max() - xf.min()) / (qmax - qmin)
        zp0 = int(np.clip(np.round(qmin - xf.min() / s_mm), qmin, qmax))

    def rel_err(s, zp):
        q = np.clip(np.round(xf / s) + zp, qmin, qmax)
        return float(np.sum((xf - s * (q - zp)) ** 2) / np.sum(xf ** 2))

    cands = np.linspace(0.05 * s_mm, 1.2 * s_mm, 400)
    errs = [rel_err(s, zp0) for s in cands]
    s_mse = float(cands[int(np.argmin(errs))])
    return s_mm, zp0, s_mse, rel_err(s_mm, zp0), rel_err(s_mse, zp0)


# 取真实激活：resnet18 layer1 的第一个 conv 输出
@torch.no_grad()
def act_of(m, x, submodule_name="layer1"):
    feats = {}
    def hook(mod, inp, out):
        feats["v"] = out.detach()
    handle = getattr(m, submodule_name).register_forward_hook(hook)
    m(x[:4])
    handle.remove()
    return feats["v"]


act = act_of(model, calib[:4], "layer1")
xf = act
s_mm_a, zp_a, s_mse_a, e_mm_a, e_mse_a = manual_scale_stats(xf, 8, symmetric=False)   # 与官方 affine 同口径
s_mm_s, zp_s, s_mse_s, e_mm_s, e_mse_s = manual_scale_stats(xf, 8, symmetric=True)    # 对称口径

obs_specs = {
    "MinMax(affine)":         dict(observer=tq.MinMaxObserver, qscheme=torch.per_tensor_affine),
    "Histogram(default act)": dict(observer=tq.HistogramObserver, qscheme=torch.per_tensor_affine),
    "MovingAvgMinMax":        dict(observer=tq.MovingAverageMinMaxObserver, qscheme=torch.per_tensor_affine),
    "MinMax(symmetric)":      dict(observer=tq.MinMaxObserver, qscheme=torch.per_tensor_symmetric),
}
rows_A = []
for name, kw in obs_specs.items():
    dt = torch.qint8 if "symmetric" in name else torch.quint8
    obs = kw["observer"](dtype=dt, qscheme=kw["qscheme"], reduce_range=False)
    obs(xf)
    sc, zp = obs.calculate_qparams()
    sc = float(sc); zpi = int(zp)
    q = torch.quantize_per_tensor(xf, sc, zpi, dt).dequantize()
    rel = float(torch.sum((xf - q) ** 2) / torch.sum(xf ** 2))
    rows_A.append(dict(observer=name, scale=sc, zero_point=zpi, rel_mse=rel))
rows_A.append(dict(observer="手撸 min-max(affine)", scale=s_mm_a, zero_point=zp_a, rel_mse=e_mm_a))
rows_A.append(dict(observer="手撸 MSE 最优(affine)", scale=s_mse_a, zero_point=zp_a, rel_mse=e_mse_a))
rows_A.append(dict(observer="手撸 min-max(sym)", scale=s_mm_s, zero_point=zp_s, rel_mse=e_mm_s))
rows_A.append(dict(observer="手撸 MSE 最优(sym)", scale=s_mse_s, zero_point=zp_s, rel_mse=e_mse_s))

log("=" * 84)
log("[A] 官方 observer vs 手撸基线（resnet18 layer1 激活，8-bit）")
log(f"{'observer':>28} {'scale':>12} {'zero_point':>11} {'rel.MSE':>12}")
for r in rows_A:
    log(f"{r['observer']:>28} {r['scale']:>12.6f} {r['zero_point']:>11d} {r['rel_mse']:>12.3e}")
log("-" * 84)
best_off = min([r for r in rows_A if "手撸" not in r["observer"]], key=lambda r: r["rel_mse"])
log(f"  官方最优：{best_off['observer']}（rel.MSE={best_off['rel_mse']:.3e}）")
log(f"  手撸 min-max(affine) {e_mm_a:.3e} / MSE 最优 {e_mse_a:.3e}："
    f"min-max 比 MSE 最优差 {10*np.log10(e_mm_a/e_mse_a):+.2f} dB")
off_mm = [r for r in rows_A if r["observer"] == "MinMax(affine)"][0]
log(f"  数值一致性：官方 MinMax(affine) scale={off_mm['scale']:.6f} vs 手撸非对称 min-max "
    f"scale={s_mm_a:.6f}（相对差 {abs(off_mm['scale']/s_mm_a-1)*100:.3f}%）")
log("  读数 1：官方 MinMax(affine) 与手撸非对称 min-max 是同一个数 —— 官方 API 没有魔法；")
log("  读数 2：官方默认激活 observer 是 Histogram，它做的是直方图上的最优裁剪，落点优于 min-max；")
log("  读数 3：权重侧官方默认 MinMax + per_channel —— 正好对上「粒度收益 > 校准收益」。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
names = [r["observer"] for r in rows_A]
vals = [r["rel_mse"] for r in rows_A]
cols = ["#4C72B0" if "手撸" not in n else "#DD8452" for n in names]
e_mm, e_mse = e_mm_a, e_mse_a
ax[0].barh(range(len(names)), vals, color=cols)
ax[0].set_yticks(range(len(names))); ax[0].set_yticklabels(names, fontsize=8)
ax[0].set_xscale("log"); ax[0].set_xlabel("rel. MSE of activation reconstruction")
ax[0].set_title("[A] Official observers vs hand-written baselines\n(blue=official, orange=manual)")
ax[0].invert_yaxis()

scales = [r["scale"] for r in rows_A]
ax[1].barh(range(len(names)), scales, color=cols)
ax[1].set_yticks(range(len(names))); ax[1].set_yticklabels(names, fontsize=8)
ax[1].set_xlabel("scale")
ax[1].set_title("[A] What scale each strategy picks")
ax[1].invert_yaxis()
savefig(fig, "official_observer_scale_and_error.png")

[A] 官方 observer vs 手撸基线（resnet18 layer1 激活，8-bit）
                    observer        scale  zero_point      rel.MSE
              MinMax(affine)     0.023561           0    7.874e-05
      Histogram(default act)     0.019017           0    5.754e-05
             MovingAvgMinMax     0.023561           0    7.874e-05
           MinMax(symmetric)     0.047123           0    3.145e-04
          手撸 min-max(affine)     0.023561           0    7.874e-05
           手撸 MSE 最优(affine)     0.018767           0    5.742e-05
             手撸 min-max(sym)     0.047309           0    3.170e-04
              手撸 MSE 最优(sym)     0.031000           0    1.814e-04
------------------------------------------------------------------------------------
  官方最优：Histogram(default act)（rel.MSE=5.754e-05）
  手撸 min-max(affine) 7.874e-05 / MSE 最优 5.742e-05：min-max 比 MSE 最优差 +1.37 dB
  数值一致性：官方 MinMax(affine) scale=0.023561 vs 手撸非对称 min-max scale=0.023561（相对差 0.000%）
  读数 1：官方 MinMax(affine) 与手撸非对称 min-max 是同一个数 —— 官方

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_observer_scale_and_error.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_observer_scale_and_error.png'

## 实验 B：官方 FakeQuantize 全流程 PTQ —— 逐层量化权重，FP32 前向评估

把每个 Conv2d/Linear 的权重交给**官方 observer 校准 + `torch.quantize_per_channel` 打包**，
再反量化回 float 填回模型，整网 FP32 前向。这样绕开"量化后端算子不全"的问题（见实验 D），
同时**完整使用官方校准语义**。指标：logits rel-MSE、top-1 一致率。

配置对比：per-tensor vs per-channel 权重、8/6/4 bit。

In [4]:
def quantize_weights_official(m, bits=8, per_channel=True):
    """用官方 observer + torch.quantize_per_channel/per_tensor 量化全部 Conv2d/Linear 权重。"""
    m2 = copy.deepcopy(m).eval()
    qmax = 2 ** (bits - 1) - 1
    info = []
    for name, mod in m2.named_modules():
        if isinstance(mod, (torch.nn.Conv2d, torch.nn.Linear)):
            W = mod.weight.detach()
            if per_channel and isinstance(mod, torch.nn.Conv2d):
                obs = tq.PerChannelMinMaxObserver(ch_axis=0, dtype=torch.qint8,
                                                  qscheme=torch.per_channel_symmetric,
                                                  quant_min=-qmax - 1, quant_max=qmax)
                obs(W)
                sc, zp = obs.calculate_qparams()
                qw = torch.quantize_per_channel(W, sc, zp, 0, torch.qint8)
            else:
                obs = tq.MinMaxObserver(dtype=torch.qint8, qscheme=torch.per_tensor_symmetric,
                                        quant_min=-qmax - 1, quant_max=qmax)
                obs(W)
                sc, zp = obs.calculate_qparams()
                qw = torch.quantize_per_tensor(W, float(sc), int(zp), torch.qint8)
            mod.weight = torch.nn.Parameter(qw.dequantize().clone())
            info.append((name, tuple(W.shape), float(sc.mean()) if sc.numel() > 1 else float(sc)))
    return m2, info


rows_B = []
for per_ch in (True, False):
    for bits in CFG["bits_grid"]:
        mq, _ = quantize_weights_official(model, bits=bits, per_channel=per_ch)
        with torch.no_grad():
            yq = logits_of(mq, evalx)
        rel = float(torch.sum((y_fp - yq) ** 2) / torch.sum(y_fp ** 2))
        cos = float(torch.nn.functional.cosine_similarity(y_fp, yq, dim=1).mean())
        agree = float((y_fp.argmax(1) == yq.argmax(1)).float().mean())
        rows_B.append(dict(scheme="per-channel" if per_ch else "per-tensor", bits=bits,
                           rel_mse=rel, cosine=cos, top1_agree=agree))

log("=" * 84)
log("[B] 官方 FakeQuantize 逐层权重量化（真实 resnet18 权重，评估为 FP32 前向）")
log(f"{'scheme':>14} {'bits':>5} {'logits rel.MSE':>16} {'cosine':>9} {'top-1 一致率':>13}")
for r in rows_B:
    log(f"{r['scheme']:>14} {r['bits']:>5} {r['rel_mse']:>16.3e} {r['cosine']:>9.4f} {r['top1_agree']:>13.3f}")
log(f"  注：样本只有 {CFG['n_eval']} 张合成图，top-1 一致率是极弱指标（FP32 的预测本身无语义）；")
log("      可靠的排序指标是 rel.MSE 与 cosine 相似度。")
for bits in CFG["bits_grid"]:
    pt = [r for r in rows_B if r["scheme"] == "per-tensor" and r["bits"] == bits][0]
    pc = [r for r in rows_B if r["scheme"] == "per-channel" and r["bits"] == bits][0]
    log(f"  {bits}-bit：per-channel 比 per-tensor 好 {10*np.log10(pt['rel_mse']/pc['rel_mse']):+.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for scheme, c in (("per-tensor", "#C44E52"), ("per-channel", "#4C72B0")):
    sub = [r for r in rows_B if r["scheme"] == scheme]
    ax[0].plot([r["bits"] for r in sub], [r["rel_mse"] for r in sub], "o-", lw=2, color=c, label=scheme)
ax[0].set_yscale("log"); ax[0].invert_xaxis()
ax[0].set_xlabel("weight bits"); ax[0].set_ylabel("logits rel. MSE")
ax[0].set_title("[B] Official calibration on real resnet18 weights")
ax[0].legend(fontsize=9)

sub = [r for r in rows_B if r["scheme"] == "per-channel"]
ax[1].bar([str(r["bits"]) for r in sub], [r["cosine"] for r in sub], color="#4C72B0")
ax[1].set_ylim(0, 1.05); ax[1].set_xlabel("weight bits"); ax[1].set_ylabel("cosine similarity to FP32 logits")
ax[1].set_title("[B] Cosine similarity (per-channel)")
for i, r in enumerate(sub):
    ax[1].text(i, r["cosine"] + 0.02, f"{r['cosine']:.3f}", ha="center", fontsize=8)
savefig(fig, "official_ptq_weight_bits.png")

[B] 官方 FakeQuantize 逐层权重量化（真实 resnet18 权重，评估为 FP32 前向）
        scheme  bits   logits rel.MSE    cosine     top-1 一致率
   per-channel     8        1.937e-03    0.9991         0.750
   per-channel     6        1.412e-02    0.9932         1.000
   per-channel     4        4.624e-01    0.7989         0.000
    per-tensor     8        4.763e-03    0.9977         1.000
    per-tensor     6        1.277e-01    0.9368         0.250
    per-tensor     4        1.325e+00    0.3358         0.000
  注：样本只有 8 张合成图，top-1 一致率是极弱指标（FP32 的预测本身无语义）；
      可靠的排序指标是 rel.MSE 与 cosine 相似度。
  8-bit：per-channel 比 per-tensor 好 +3.91 dB
  6-bit：per-channel 比 per-tensor 好 +9.57 dB
  4-bit：per-channel 比 per-tensor 好 +4.57 dB
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_ptq_weight_bits.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_ptq_weight_bits.png'

## 实验 C：真实存储开销 —— int8 到底省了多少字节？

用 `torch.quantize_per_channel(...).int_repr()` 拿**真正的 int8 张量**，统计字节数，
并把 scale/zero_point 的元数据税一起算进去（01 篇 §7.2 的"scale 元数据税"在真实模型上的数字）。

In [5]:
def pack_weight_stats(W, bits=8, granularity="per-channel"):
    """用官方 observer 取 scale，再按 bits 真正打包。

    注意：torch.ao 经典栈的量化张量类型只有 qint8/quint8（8-bit），
    **4/6-bit 必须自己做位打包** —— 这正是官方 API 与"LLM 量化部署"之间的落差。
    """
    qmax = 2 ** (bits - 1) - 1
    if granularity == "per-channel":
        obs = tq.PerChannelMinMaxObserver(ch_axis=0, dtype=torch.qint8,
                                          qscheme=torch.per_channel_symmetric,
                                          quant_min=-qmax - 1, quant_max=qmax)
        obs(W); sc, zp = obs.calculate_qparams()
        dq = torch.quantize_per_channel(W, sc, zp, 0, torch.qint8).dequantize()
        n_scale = sc.numel()
    else:
        obs = tq.MinMaxObserver(dtype=torch.qint8, qscheme=torch.per_tensor_symmetric,
                                quant_min=-qmax - 1, quant_max=qmax)
        obs(W); sc, zp = obs.calculate_qparams()
        dq = torch.quantize_per_tensor(W, float(sc), int(zp), torch.qint8).dequantize()
        n_scale = 1
    codes = torch.clamp(torch.round(W / sc.reshape([-1] + [1] * (W.dim() - 1))
                                    if sc.numel() > 1 else torch.round(W / float(sc))),
                        -qmax - 1, qmax)
    payload = pack_bytes(codes.flatten().numpy().astype(np.int64), bits)
    meta = n_scale * 4                      # fp32 scale（对称量化 zp=0，不占字节）
    rel = float(torch.sum((W - dq) ** 2) / torch.sum(W ** 2))
    return payload, meta, rel


def pack_bytes(codes, bits):
    """把 int code 按 bits 位打包，返回真实字节数（8-bit 时就是元素数）。"""
    if bits == 8:
        return int(codes.size)
    vals = codes & ((1 << bits) - 1)
    stream = np.zeros(vals.size * bits, dtype=np.uint8)
    for i in range(bits):
        stream[i::bits] = ((vals >> (bits - 1 - i)) & 1).astype(np.uint8)
    return int(np.ceil(stream.size / 8))


rows_C = []
Wref = model.layer1[0].conv1.weight.detach()      # (64, 64, 3, 3)
fp_bytes_w = Wref.numel() * 4
for bits in CFG["bits_grid"]:
    for gran in ("per-tensor", "per-channel"):
        payload, meta, rel = pack_weight_stats(Wref, bits, gran)
        rows_C.append(dict(bits=bits, granularity=gran, fp_bytes=fp_bytes_w, payload=payload,
                           meta=meta, total=payload + meta, compress=fp_bytes_w / (payload + meta),
                           rel_mse=rel))

log("=" * 84)
log(f"[C] 真实存储（layer1.0.conv1，{tuple(Wref.shape)}，{Wref.numel()} 参数，含位打包）")
log(f"{'bits':>5} {'granularity':>14} {'FP32':>9} {'payload':>9} {'meta':>7} {'合计':>9} {'压缩比':>8} {'rel.MSE':>11}")
for r in rows_C:
    log(f"{r['bits']:>5} {r['granularity']:>14} {r['fp_bytes']:>9} {r['payload']:>9} "
        f"{r['meta']:>7} {r['total']:>9} {r['compress']:>7.2f}x {r['rel_mse']:>11.3e}")
pc8 = [r for r in rows_C if r["bits"] == 8 and r["granularity"] == "per-channel"][0]
log(f"  元数据税（per-channel 8-bit）：{pc8['meta']/pc8['total']*100:.2f}%")
log("  要点：torch.ao 经典栈只有 qint8/quint8 张量类型，4/6-bit 必须自己做 bit-packing；")
log("        位宽越低、粒度越细，scale 元数据税越重 —— 这正是 01 篇 §7.2 的 16/g bit per element。")

# 元数据税随粒度扫描（4-bit，固定权重）
tax_rows = []
flat = Wref.flatten().numel()
for name, g in (("per-tensor", None), ("per-channel(64)", 576), ("per-group(64)", 64),
                ("per-group(32)", 32), ("per-group(16)", 16)):
    n_scale = 1 if g is None else int(np.ceil(flat / g))
    payload = pack_bytes(np.zeros(flat, dtype=np.int64), 4)
    meta = n_scale * 4
    tax_rows.append(dict(granularity=name, n_scale=n_scale, payload=payload, meta=meta,
                         total=payload + meta, compress=fp_bytes_w / (payload + meta),
                         tax_pct=100 * meta / (payload + meta)))
log("-" * 84)
log("[C] 4-bit 下元数据税随粒度（同一层）")
log(f"{'granularity':>16} {'#scale':>7} {'payload':>9} {'meta':>8} {'压缩比':>8} {'元数据税':>9}")
for r in tax_rows:
    log(f"{r['granularity']:>16} {r['n_scale']:>7} {r['payload']:>9} {r['meta']:>8} "
        f"{r['compress']:>7.2f}x {r['tax_pct']:>8.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
w = 0.36; xs = np.arange(len(CFG["bits_grid"]))
for i, gran in enumerate(("per-tensor", "per-channel")):
    sub = [r for r in rows_C if r["granularity"] == gran]
    ax[0].bar(xs + (i - 0.5) * w, [r["compress"] for r in sub], width=w, label=gran)
ax[0].set_xticks(xs); ax[0].set_xticklabels([str(b) for b in CFG["bits_grid"]])
ax[0].set_xlabel("weight bits"); ax[0].set_ylabel("compression ratio (FP32 bytes / total)")
ax[0].set_title("[C] Real compression incl. scale metadata"); ax[0].legend(fontsize=9)

ax[1].barh(range(len(tax_rows)), [r["tax_pct"] for r in tax_rows], color="#DD8452")
ax[1].set_yticks(range(len(tax_rows)))
ax[1].set_yticklabels([r["granularity"] for r in tax_rows], fontsize=8)
ax[1].set_xlabel("metadata tax (% of stored bytes)")
ax[1].set_title("[C] 4-bit: finer granularity = heavier scale tax")
ax[1].invert_yaxis()
for i, r in enumerate(tax_rows):
    ax[1].text(r["tax_pct"] + 0.3, i, f"{r['tax_pct']:.1f}%", va="center", fontsize=8)
savefig(fig, "official_storage_and_metadata_tax.png")

[C] 真实存储（layer1.0.conv1，(64, 64, 3, 3)，36864 参数，含位打包）
 bits    granularity      FP32   payload    meta        合计      压缩比     rel.MSE
    8     per-tensor    147456     36864       4     36868    4.00x   1.009e-03
    8    per-channel    147456     36864     256     37120    3.97x   2.424e-04
    6     per-tensor    147456     27648       4     27652    5.33x   1.637e-02
    6    per-channel    147456     27648     256     27904    5.28x   3.983e-03
    4     per-tensor    147456     18432       4     18436    8.00x   2.224e-01
    4    per-channel    147456     18432     256     18688    7.89x   6.769e-02
  元数据税（per-channel 8-bit）：0.69%
  要点：torch.ao 经典栈只有 qint8/quint8 张量类型，4/6-bit 必须自己做 bit-packing；
        位宽越低、粒度越细，scale 元数据税越重 —— 这正是 01 篇 §7.2 的 16/g bit per element。
------------------------------------------------------------------------------------
[C] 4-bit 下元数据税随粒度（同一层）
     granularity  #scale   payload     meta      压缩比      元数据税
      per-tensor       1     18432        4  

[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_storage_and_metadata_tax.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_storage_and_metadata_tax.png'

## 实验 D：真 int8 算子 + 延迟 —— 以及 torch 2.10 在 arm64 macOS 上的真实边界

这里做一次**完整的 `prepare → calibrate → convert → int8 前向`**，并实测延迟与体积。
用一个**没有残差 add** 的小 CNN（Conv+ReLU+Pool+FC），原因是：

> qnnpack 后端在本机**没有 `aten::add.out` 的 QuantizedCPU 实现**，所以 resnet18 的残差加法
> 在 convert 后会直接抛 `NotImplementedError`；正确做法是把 add 换成
> `torch.ao.quantization.FloatFunctional`（torchvision 的量化版 ResNet 就是这么改的）。

同时记录几条真实情报（对博客 E3 有用）：

- `torch.backends.quantized.supported_engines == ['qnnpack']`，且**必须显式
  `torch.backends.quantized.engine = 'qnnpack'`**，否则 `convert` 报 `NoQEngine`
- Linear 的 **dynamic** 量化（`quantize_dynamic`）在本机**不可用**：`quantized::linear_prepack NoQEngine`
- torch 2.10 已经把 `torch.ao.quantization` 标为 **deprecated**，官方指向 **torchao**

In [6]:
class SmallCNN(torch.nn.Module):
    """无残差 add 的小 CNN：qnnpack 能完整 convert。"""
    def __init__(self, n_cls=10):
        super().__init__()
        self.quant = torch.ao.quantization.QuantStub()
        self.conv1 = torch.nn.Conv2d(3, 16, 3, padding=1)
        self.relu1 = torch.nn.ReLU()
        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = torch.nn.ReLU()
        self.pool = torch.nn.AdaptiveAvgPool2d((4, 4))
        self.fc = torch.nn.Linear(32 * 4 * 4, n_cls)
        self.dequant = torch.ao.quantization.DeQuantStub()

    def forward(self, x):
        x = self.quant(x)
        x = self.relu1(self.conv1(x))
        x = torch.nn.functional.max_pool2d(self.relu2(self.conv2(x)), 2)
        x = self.pool(x)
        x = self.dequant(self.fc(x.flatten(1)))
        return x


base = SmallCNN().eval()
base.qconfig = tq.get_default_qconfig("qnnpack")
prepared = tq.prepare(base, inplace=False)
with torch.no_grad():
    for i in range(0, calib.shape[0], 4):
        prepared(calib[i:i + 4])
converted = tq.convert(prepared, inplace=False)
log("[D] convert 成功：conv1 = " + type(converted.conv1).__module__ + "." + type(converted.conv1).__name__)

xs = evalx[:CFG["n_eval"]]
with torch.no_grad():
    y0 = base(xs); y1 = converted(xs)
rel_d = float(torch.sum((y0 - y1) ** 2) / torch.sum(y0 ** 2))
b_fp = state_dict_bytes(base); b_int = state_dict_bytes(converted)
t_fp = bench(base, xs); t_int = bench(converted, xs)
log("=" * 84)
log("[D] 真 int8 算子（SmallCNN，qnnpack，CPU）")
log(f"  logits rel.MSE      = {rel_d:.3e}")
log(f"  state_dict 体积      = FP32 {b_fp/1024:.1f} KB  ->  INT8 {b_int/1024:.1f} KB（{b_fp/b_int:.2f}x）")
log(f"  前向延迟（batch={CFG['batch']}）= FP32 {t_fp:.2f} ms  ->  INT8 {t_int:.2f} ms（{t_fp/t_int:.2f}x）")
log("  读数：Conv 路径能拿到真实加速；但这是无残差的小网，量级不能外推到 resnet18。")

# ---- 记录官方栈在本机的真实边界（诚实清单）
probes = {}
try:
    m = torch.nn.Sequential(torch.nn.Linear(64, 32))
    tq.quantize_dynamic(m, {torch.nn.Linear}, dtype=torch.qint8)
    probes["quantize_dynamic(Linear)"] = "OK"
except Exception as e:
    probes["quantize_dynamic(Linear)"] = "FAILED: " + str(e)[:60]
try:
    w = copy.deepcopy(model); w.eval()
    w.qconfig = tq.get_default_qconfig("qnnpack")
    wp = tq.prepare(w, inplace=False)
    with torch.no_grad(): wp(calib[:2])
    wc = tq.convert(wp, inplace=False)
    with torch.no_grad(): wc(calib[:1])
    probes["resnet18 static convert"] = "OK"
except Exception as e:
    probes["resnet18 static convert"] = "FAILED: " + str(e)[:80]
log("-" * 84)
log("[D] 本机（arm64 macOS, torch %s）官方量化栈可用性清单：" % torch.__version__)
for k, v in probes.items():
    log(f"  - {k:<28} {v}")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax[0].bar(["FP32", "INT8"], [b_fp / 1024, b_int / 1024], color=["#C44E52", "#4C72B0"])
ax[0].set_ylabel("state_dict size (KB)")
ax[0].set_title(f"[D] Real model size ({b_fp/b_int:.2f}x)")
for i, v in enumerate([b_fp / 1024, b_int / 1024]):
    ax[0].text(i, v, f"{v:.1f} KB", ha="center", va="bottom", fontsize=9)
ax[1].bar(["FP32", "INT8"], [t_fp, t_int], color=["#C44E52", "#4C72B0"])
ax[1].set_ylabel(f"latency (ms, batch={CFG['batch']})")
ax[1].set_title(f"[D] Real latency ({t_fp/t_int:.2f}x)")
for i, v in enumerate([t_fp, t_int]):
    ax[1].text(i, v, f"{v:.2f} ms", ha="center", va="bottom", fontsize=9)
savefig(fig, "official_int8_size_and_latency.png")

[D] convert 成功：conv1 = torch.ao.nn.quantized.modules.conv.Conv2d


[D] 真 int8 算子（SmallCNN，qnnpack，CPU）
  logits rel.MSE      = 2.812e-03
  state_dict 体积      = FP32 42.6 KB  ->  INT8 15.2 KB（2.81x）
  前向延迟（batch=4）= FP32 44.38 ms  ->  INT8 39.34 ms（1.13x）
  读数：Conv 路径能拿到真实加速；但这是无残差的小网，量级不能外推到 resnet18。


------------------------------------------------------------------------------------
[D] 本机（arm64 macOS, torch 2.10.0）官方量化栈可用性清单：
  - quantize_dynamic(Linear)     OK
  - resnet18 static convert      FAILED: Could not run 'quantized::conv2d.new' with arguments from the 'CPU' backend. Thi
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_int8_size_and_latency.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/official_int8_size_and_latency.png'

## 结论汇总

In [7]:
summary = {
    "meta": dict(mode=MODE, cfg=CFG, torch=torch.__version__, seed=SEED,
                 weights=WEIGHTS, n_params=int(n_params),
                 note="real resnet18 weights + synthetic images; volumes/latency/scales are real metrics, logits are relative-only"),
    "A_observers": rows_A,
    "B_official_ptq": rows_B,
    "C_storage": [{k: (float(v) if isinstance(v, (int, float)) else v) for k, v in r.items()} for r in rows_C],
    "D_int8_operator": dict(rel_mse=rel_d, bytes_fp=int(b_fp), bytes_int8=int(b_int),
                            ms_fp=float(t_fp), ms_int8=float(t_int),
                            speedup=float(t_fp / t_int), availability=probes),
}
best8 = [r for r in rows_B if r["bits"] == 8 and r["scheme"] == "per-channel"][0]
log("")
log("=" * 84)
log("结论汇总")
log("=" * 84)
log(f"1) [A] 官方默认激活 observer 是 Histogram（rel.MSE={[r for r in rows_A if r['observer'].startswith('Histogram')][0]['rel_mse']:.3e}）；"
    f"手撸 min-max 比 MSE 最优差 {10*np.log10(e_mm/e_mse):+.2f} dB")
log(f"2) [B] 8-bit per-channel 权重量化：logits rel.MSE={best8['rel_mse']:.3e}，"
    f"与 FP32 的 top-1 一致率 {best8['top1_agree']:.3f}")
pc = [r for r in rows_B if r["scheme"] == "per-channel"][0]
pt = [r for r in rows_B if r["scheme"] == "per-tensor" and r["bits"] == pc["bits"]][0]
log(f"3) [B] per-channel 比 per-tensor 好 {10*np.log10(pt['rel_mse']/pc['rel_mse']):+.2f} dB（{pc['bits']}-bit，真实权重）")
log(f"4) [C] 真实压缩比（含 scale 元数据）：8-bit per-channel {pc8['compress']:.2f}x，"
    f"元数据税 {pc8['meta']/pc8['total']*100:.2f}%")
log(f"5) [D] 真 int8 算子：体积 {b_fp/b_int:.2f}x、延迟 {t_fp/t_int:.2f}x（SmallCNN，CPU，qnnpack）")
log("6) [D] 本机边界（实测，见上方清单）：" + "；".join(f"{k}={v[:28]}" for k, v in probes.items()))
log("   另：torch.ao.quantization 在 2.10 已 deprecated，官方迁移目标为 torchao")
log("=" * 84)

with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log(f"[save] {os.path.join(RES, 'results.json')}")
log(f"[save] {os.path.join(RES, 'stdout.txt')}")


结论汇总
1) [A] 官方默认激活 observer 是 Histogram（rel.MSE=5.754e-05）；手撸 min-max 比 MSE 最优差 +1.37 dB
2) [B] 8-bit per-channel 权重量化：logits rel.MSE=1.937e-03，与 FP32 的 top-1 一致率 0.750
3) [B] per-channel 比 per-tensor 好 +3.91 dB（8-bit，真实权重）
4) [C] 真实压缩比（含 scale 元数据）：8-bit per-channel 3.97x，元数据税 0.69%
5) [D] 真 int8 算子：体积 2.81x、延迟 1.13x（SmallCNN，CPU，qnnpack）
6) [D] 本机边界（实测，见上方清单）：quantize_dynamic(Linear)=OK；resnet18 static convert=FAILED: Could not run 'quant
   另：torch.ao.quantization 在 2.10 已 deprecated，官方迁移目标为 torchao
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/results.json
[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/official_torchao_ptq/results/stdout.txt


## 与手撸目录的分工

| 目录 | 实现 | 回答的问题 |
|---|---|---|
| `rtn_llmint8/`、`quantizer_granularity/` | 纯 numpy 手撸 | 算法机理、公式、梯度、可控消融 |
| 本目录 `official_torchao_ptq/` | PyTorch 官方栈 | 官方 API 的默认行为、真实体积/延迟、真实边界（哪个 API 在本机跑不通） |
| `official_torchao_qat/` | PyTorch 官方栈 | `prepare_qat`/`convert` 全流程 + 自定义 LSQ/PACT FakeQuantize |
| `official_llm_ptq_torch/` | transformers + torch | 真实 LLM 上的 RTN/GPTQ/SmoothQuant/AWQ |